# 第七课:Multi-Head Attention

讲义:[07-multi-head.md](07-multi-head.md)。切维度 → 并行注意力 → 拼接。

In [ ]:
import numpy as np, math

def softmax_rows(M):
    m = M.max(axis=-1, keepdims=True)
    e = np.exp(M - m)
    return e / e.sum(axis=-1, keepdims=True)

rng = np.random.default_rng(0)
X = rng.normal(size=(3, 8))      # 3 词 × 8 维
H, d = 2, 8
dh = d // H

heads = []
for h in range(H):
    Xh = X[:, h*dh:(h+1)*dh]                          # 切出该头子空间
    W = softmax_rows(Xh @ Xh.T / math.sqrt(dh))       # 每头独立注意力
    heads.append(W @ Xh)
concat = np.concatenate(heads, axis=-1)

print('每头输出 shape:', [h.shape for h in heads], ' concat:', concat.shape)
print('头1 第1词:', np.round(concat[0, :4], 3))
print('头2 第1词:', np.round(concat[0, 4:], 3))
single = (softmax_rows(X @ X.T / math.sqrt(8)) @ X)[0, :4]
print('单头 第1词:', np.round(single, 3), '← 三组各不相同')

## 完整版:带 W 投影的多头(讲义 §4)

In [ ]:
def multi_head_attention(X, Wq, Wk, Wv, Wo, H):
    n, d = X.shape
    dh = d // H
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    heads = []
    for h in range(H):
        q, k, v = (M[:, h*dh:(h+1)*dh] for M in (Q, K, V))
        W = softmax_rows(q @ k.T / math.sqrt(dh))
        heads.append(W @ v)
    return np.concatenate(heads, axis=-1) @ Wo

rng = np.random.default_rng(1)
Wq, Wk, Wv, Wo = (rng.normal(size=(8, 8)) * 0.3 for _ in range(4))
out = multi_head_attention(X, Wq, Wk, Wv, Wo, H=2)
print('输出 shape:', out.shape)
print('注意力参数 4*d^2 (d=512):', 4 * 512**2)

## 作业现场

- C2:reshape/transpose 快速版,与 for 版逐元素对照。
- C3:GQA 版(KV 头少于 Q 头)。
- D1:KV Cache 字节数计算器。

In [ ]:
# TODO(C2): 快速版 —— reshape + transpose,不写 for
def multi_head_attention_fast(X, Wq, Wk, Wv, Wo, H):
    # 提示: Q.reshape(n, H, dh).transpose(1, 0, 2) → [H, n, dh]
    # 批量算所有头的 scores: [H, n, n],softmax(axis=-1),再 [H, n, dh]
    # 最后 transpose(1,0,2).reshape(n, d) @ Wo
    ...

# 对照:
# out_fast = multi_head_attention_fast(X, Wq, Wk, Wv, Wo, H=2)
# print('逐元素相等:', np.allclose(out_fast, out))

# TODO(D1): KV Cache 计算器
def kv_cache_bytes(n_layers, n_kv_heads, d_h, seq_len, bytes_per=2):
    # 你的代码:2(K和V) × 层数 × KV头数 × d_h × seq_len × bytes
    ...

# Llama-3-8B: 32层, 8 KV头, d_h=128, 8K上下文, bf16
# print('每 token:', kv_cache_bytes(32, 8, 128, 1), 'B')
# print('8K 上下文:', kv_cache_bytes(32, 8, 128, 8192) / 1024**3, 'GB')